Goal: Rebuild legacy R code to get the WMC dashboard metrics using the .csv files generated from main

In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [2]:
# Keep month/year inputs simple for quick reruns
year = "25"
month = "10"

# set working directories - can change in future, be careful
root_dir = "K:/AP/TTM/"
busstate_dir = os.path.join(root_dir, "Data/WMC Dashboard/BusState Cleaned") # Where cleaned busstate data is stored
stops_dir = os.path.join(root_dir, "Data/WMC Dashboard/stops") # where a copy of previous stops data is stored
# NOTE: stops data WILL need to be updated with new med center stops

# pull in necessary static files NOTE: again these WILL need to be updated with new med center stops - may break things downstream so be careful after changing
stop_inventory = pd.read_csv(os.path.join(stops_dir, "stop_inventory.csv"))
pattern_stops = pd.read_csv(os.path.join(stops_dir, "pattern_stops.csv"), header=None)

# have to clean up year/month for proper file reading
# simple map for month number to month abbreviation for file reading
month_abbrev_map = {"01": "JAN", "02": "FEB", "03": "MAR", "04": "APR", "05": "MAY", "06": "JUN", "07": "JUL", "08": "AUG", "09": "SEP", "10": "OCT", "11": "NOV", "12": "DEC"}

year_full = 2000 + int(year)
month_full = month_abbrev_map[month]

# pull in cleaned busstate data for month and year of interest
busstate = pd.read_csv(os.path.normpath(os.path.join(busstate_dir, f"{year_full}-{month_full}-busstate.csv")))

In [3]:
# set directory to save dashbaord data
dashboard_dir = os.path.join(root_dir, f"Data/WMC Dashboard/Dashboard Data/{year_full}/{month_full}")
Path(dashboard_dir).mkdir(parents=True, exist_ok=True)

In [4]:
# original R code selected the 1st and 10th cols
pattern_stops_filtered = pattern_stops.iloc[:, [0, 9]].drop_duplicates()
# name schema per original R code
pattern_stops_filtered.columns = ["ROUTE", "STOP_ID"]

# left join stop inventory 
stops_df = pattern_stops_filtered.merge(stop_inventory, on="STOP_ID", how="left")

In [7]:
# stops.info()
# sum(stops['ROUTE'] == "MC")

# print(stops[stops['ROUTE'] == "MC"])
#stops_df.info()

In [5]:
# create distance function
# NOTE: This does NOT account for curvature of the earth, but locations are close enough that it is negligible
# NOTE: If we want to be more precise in the future, this function can be updated.
def distance(x1, x2, y1, y2):
    '''
    Calculate the distance between two points.

    Args:
        x1 (float): The x-coordinate of the first point longitude.
        x2 (float): The x-coordinate of the second point longitude.
        y1 (float): The y-coordinate of the first point latitude.
        y2 (float): The y-coordinate of the second point latitude.
    Returns:
        float: The distance between the two points.
    '''
    return np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

In [6]:
# Create function to determine the closest stop
def which_stop(lat, lon, route="MC", stops=stops_df):
    '''
    Determine the closest stop to a given latitude and longitude. Med center route is default, but can be updated to other routes as needed. 
    NOTE: This will potentially break when new stops are added start Dec. 2025 - be careful when updating stop inventory.

    Args:
        lat (float): The latitude of the point of interest.
        lon (float): The longitude of the point of interest.
        route (str): The route to filter stops by. Default is "MC" for medical center. Based on ROUTE col in stops DataFrame.
        stops (DataFrame): A DataFrame containing stop information, including 'STOP_ID', 'LAT', and 'LONG' columns.

    Returns:
        int: The STOP_ID of the closest stop.
    '''
    # isolate the specified route stops
    selected_stops = stops[stops['ROUTE'] == route].copy()

    # Calculate distance to each stop
    selected_stops['DISTANCE'] = distance(lon, selected_stops['LONG'], lat, selected_stops['LAT'])

    # subsetting only stops within 0.0005 units
    selected_stops = selected_stops[selected_stops['DISTANCE'] < 0.0005] # do not know exact units, but this was the threshold used in original R code
    selected_stops = selected_stops.sort_values('DISTANCE')

    # sorted by distance, therfore return first stop which is closest
    if not selected_stops.empty:
        return selected_stops.iloc[0]["STOP_ID"]
    else:
        return None


In [7]:
# which_stop(39.997570, -83.018117)

busstate.info()
busstate.head()

#which_stop(40, -80)

<class 'pandas.DataFrame'>
RangeIndex: 1089062 entries, 0 to 1089061
Data columns (total 23 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   DATE                    1089062 non-null  str    
 1   BUS_ID                  1089062 non-null  int64  
 2   RUN_ID                  1083282 non-null  float64
 3   DEST_SIGN_ROUTE_TEXT    1089046 non-null  str    
 4   BLOCK_ID                1087777 non-null  float64
 5   TRIP_ID                 1080866 non-null  float64
 6   ROUTE_ID                1084654 non-null  str    
 7   STOP_SEQUENCE           1089062 non-null  int64  
 8   LATITUDE                1089062 non-null  float64
 9   LONGITUDE               1089062 non-null  float64
 10  HEADING                 1089062 non-null  int64  
 11  OPERATOR_ID             1072166 non-null  float64
 12  ODOMETER_DISTANCE       1089062 non-null  int64  
 13  TIMEPOINT_ID            121084 non-null   str    
 14  EVENT_TYPE   

,DATE,BUS_ID,RUN_ID,DEST_SIGN_ROUTE_TEXT,BLOCK_ID,TRIP_ID,ROUTE_ID,STOP_SEQUENCE,LATITUDE,LONGITUDE,...,TIMEPOINT_ID,EVENT_TYPE,EVENT_TIME,BOARDINGS,ALIGHTINGS,PASSENGER_LOAD,TRIP_START_TIME,DEPARTURE_TIME,ENTER_STOP_WINDOW_TIME,EXIT_STOP_WINDOW_TIME
0,2025-10-01,1303,1503.0,MC,23675102.0,3733020.0,MC05,3,40.002903,-83.041008,...,NaN,9,00:00:00,0,0,0,23:55:43,NaN,NaN,NaN
1,2025-10-01,1303,1503.0,MC,23675102.0,3733020.0,MC05,3,40.002907,-83.041039,...,NaN,10,00:00:01,0,0,0,23:55:43,NaN,NaN,NaN
2,2025-10-01,1906,2201.0,SS,23674802.0,2300020.0,SS02,2,40.016163,-83.029213,...,NaN,2,00:00:01,0,0,0,23:54:47,NaN,NaN,NaN
3,2025-10-01,1502,1502.0,MC,23675202.0,22020.0,MC05,4,39.999454,-83.028168,...,NaN,2,00:00:03,0,0,0,23:52:34,NaN,NaN,NaN
4,2025-10-01,1501,1501.0,MC,23675302.0,167020.0,MC02,1,39.997444,-83.018684,...,NaN,2,00:00:05,0,0,0,23:57:02,NaN,NaN,NaN


In [ ]:
# Function to process the busstate data for the medical center
def process_mc_busstate(busstate_df):
    
    processed = (
        busstate_df

        # filter for MC runs - within 1500 and 1600
        .loc[(busstate_df['RUN_ID'] >= 1500) & (busstate_df['RUN_ID'] < 1600)]

        # convert time
        .assign(
            EVENT_TIME = pd.to_datetime(busstate_df['EVENT_TIME'], format="%H:%M:%S"),
            DEPARTURE_TIME = pd.to_datetime(busstate_df['DEPARTURE_TIME'], format="%H:%M:%S"),
            ENTER_STOP_WINDOW_TIME = pd.to_datetime(busstate_df['ENTER_STOP_WINDOW_TIME'], format="%H:%M:%S"),
            EXIT_STOP_WINDOW_TIME = pd.to_datetime(busstate_df['EXIT_STOP_WINDOW_TIME'], format="%H:%M:%S")
        )

        # arrange by date and even time
        .sort_values(['DATE', 'EVENT_TIME'])

        # Assign stop ID
        # NOTE: Might need to edit this function to address not being at a stop?
        .assign(
            STOP=lambda df: df.apply(lambda row: which_stop(row['LATITUDE'], row['LONGITUDE'], stops=stops_df), axis=1)
        )

        # filter out rows where stop was not assigned
        .loc[lambda df: df['STOP'].notna()] 

        # convert STOP to numeric
        .assign(STOP = lambda df: pd.to_numeric(df['STOP'], errors='coerce'))
        
        # join with stop inventory
        .merge(stop_inventory, how='left', left_on='STOP', right_on='STOP_ID')
        
        # remove dummy stops - R code has 27 and 461 as inbound and outbound dummy stops.
        .loc[lambda df: ~df['STOP'].isin([27, 461])]
        
        # sort by BUS_ID, DATE, EVENT_TIME for elapsed calculations
        .sort_values(['BUS_ID', 'DATE', 'EVENT_TIME'])
        
        # convert times to minutes
        .assign(
            EVENT_TIME_MIN = lambda df: df['EVENT_TIME'].dt.hour * 60 + df['EVENT_TIME'].dt.minute + df['EVENT_TIME'].dt.second / 60,
            DEPARTURE_TIME_MIN = lambda df: df['DEPARTURE_TIME'].dt.hour * 60 + df['DEPARTURE_TIME'].dt.minute + df['DEPARTURE_TIME'].dt.second / 60,
            ENTER_STOP_WINDOW_TIME_MIN = lambda df: df['ENTER_STOP_WINDOW_TIME'].dt.hour * 60 + df['ENTER_STOP_WINDOW_TIME'].dt.minute + df['ENTER_STOP_WINDOW_TIME'].dt.second / 60,
            EXIT_STOP_WINDOW_TIME_MIN = lambda df: df['EXIT_STOP_WINDOW_TIME'].dt.hour * 60 + df['EXIT_STOP_WINDOW_TIME'].dt.minute + df['EXIT_STOP_WINDOW_TIME'].dt.second / 60
        )
        
        # compute elapsed time between rows
        .assign(ELAPSED = lambda df: (df['EVENT_TIME_MIN'] - df['EVENT_TIME_MIN'].shift(1)).round(2))
    )

    # Group and count logic - previously fixed R code - keep as redundant for now, but may be able to simplify in the future
    processed = processed.assign(
        NEW_GROUP = lambda df: (
            df['ELAPSED'].isna() |
            df['STOP'].isna() |
            df['STOP'].shift(1).isna() |
            df['BUS_ID'].isna() |
            df['BUS_ID'].shift(1).isna() |
            (df['STOP'] != df['STOP'].shift(1)) |
            (df['BUS_ID'] != df['BUS_ID'].shift(1)) |
            (df['ELAPSED'] >= 60)
        ).astype(int),
        COUNT = lambda df: df['NEW_GROUP'].cumsum()
    )

    # no longer needed
    processed = processed.drop(columns=['NEW_GROUP'])

    # get rid of non existing even times
    processed = processed.loc[processed['EVENT_TIME'].notna()]

    # consolidate ridership
    consolidated = (
        processed

        .groupby(['BUS_ID', 'DATE', 'COUNT', 'STOP_NAME', 'STOP', 'RUN_ID'], as_index=False)

        # Summary statistics
        .agg(
            BOARDINGS = ('BOARDINGS', 'sum'),
            ALIGHTINGS = ('ALIGHTINGS', 'sum'),
            LOAD = ('PASSENGER_LOAD', 'max'),
            EARLY_EVENT=('EVENT_TIME_MIN', 'min'),
            LATE_EVENT=('EVENT_TIME_MIN', 'max'),
            DEPARTURE_TIME=('DEPARTURE_TIME_MIN', 'max'),
            ENTER_STOP=('ENTER_STOP_WINDOW_TIME_MIN', 'min'),
            EXIT_STOP=('EXIT_STOP_WINDOW_TIME_MIN', 'max'),
            DEST=('DEST_SIGN_ROUTE_TEXT', 'last')
        )

        # Arrival and departure logic - from R code
        .assign(
            ARRIVAL = lambda df: df[['EARLY_EVENT', 'ENTER_STOP']].min(axis=1),
            DEPARTURE = lambda df: df[['LATE_EVENT', 'DEPARTURE_TIME', 'EXIT_STOP']].max(axis=1),
            DWELL = lambda df: df['DEPARTURE'] - df['ARRIVAL']
        )

        # drop extra time cols
        .drop(columns=[
            'EARLY_EVENT',
            'LATE_EVENT',
            'DEPARTURE_TIME',
            'ENTER_STOP',
            'EXIT_STOP'
        ])

        # assign hour and min
        .assign(
            HOUR=lambda df: np.floor(df['ARRIVAL'] / 60),
            MIN=lambda df: np.floor(df['ARRIVAL'] % 60) # modulus to get remaining min after hour
        )
    )

    # clean infinities - previously fixed r code,  may be able to change in future
    consolidated[['ARRIVAL', 'DEPARTURE', 'DWELL']] = (
        consolidated[['ARRIVAL', 'DEPARTURE', 'DWELL']]
        .replace([np.inf, -np.inf], np.nan)
    )

    return consolidated

In [9]:
mc_busstate = process_mc_busstate(busstate)
# should take approximately 7-8 minutes to run for 1 month of data

In [10]:
mc_busstate.info() # NOTE: R dataframe at this point has 35035 obersvations - we have 33636 obs
mc_busstate.tail(5)


<class 'pandas.DataFrame'>
RangeIndex: 33636 entries, 0 to 33635
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   BUS_ID      33636 non-null  int64  
 1   DATE        33636 non-null  str    
 2   COUNT       33636 non-null  int64  
 3   STOP_NAME   33636 non-null  str    
 4   STOP        33636 non-null  float64
 5   RUN_ID      33636 non-null  float64
 6   BOARDINGS   33636 non-null  int64  
 7   ALIGHTINGS  33636 non-null  int64  
 8   LOAD        33636 non-null  int64  
 9   DEST        33636 non-null  str    
 10  ARRIVAL     33636 non-null  float64
 11  DEPARTURE   33636 non-null  float64
 12  DWELL       33636 non-null  float64
 13  HOUR        33636 non-null  float64
 14  MIN         33636 non-null  float64
dtypes: float64(7), int64(5), str(3)
memory usage: 3.8 MB


,BUS_ID,DATE,COUNT,STOP_NAME,STOP,RUN_ID,BOARDINGS,ALIGHTINGS,LOAD,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MIN
33631,2503,2025-10-30,33597,CARMACK 2,403.0,1507.0,1,9,27,MC,1190.616667,1191.500000,0.883333,19.0,50.0
33632,2503,2025-10-30,33598,CARMACK 3,404.0,1507.0,0,3,24,MC,1192.083333,1194.116667,2.033333,19.0,52.0
33633,2503,2025-10-30,33599,CARMACK 5 + STOP 1,94.0,1507.0,0,2,22,MC,1194.983333,1195.483333,0.500000,19.0,54.0
33634,2503,2025-10-30,33600,CARMACK 5 + STOP 2,95.0,1507.0,2,1,23,MC,1196.000000,1196.500000,0.500000,19.0,56.0
33635,2503,2025-10-30,33601,JOHN HERRICK LOOP,405.0,1507.0,9,4,28,MC,1201.350000,1204.300000,2.950000,20.0,1.0


In [11]:
# consolidate calculatign headways for each stop into a single function
def calculate_headways(df, stop_num):
    '''
    Calculate the headways for the given stop number. Key for stop numbers is in stop_inventory.csv file.
    NOTE: This should work with any stop with cleaned busstate data from processing.py

    Args:
        df (DataFrame): A pandas DataFrame containing processed busstate data.
        stop_num (int): The stop number of the stop to calculate the headways for.
    Returns:
        DataFrame: A pandas DataFrame containing the headways for the specified stop.
    '''
    df_hw = (
        df
        .loc[(df["STOP"] == stop_num) & (df["ARRIVAL"].notna())]
        .sort_values(['DATE', 'ARRIVAL'])
        .copy()
    )

    df_hw["HEADWAY"] = df_hw.groupby("DATE")["ARRIVAL"].diff()

    df_hw['DATE'] = pd.to_datetime(df_hw['DATE'])

    #NOTE: Eventually change this to convert midnight BEFORE headway>?
    df_hw['DATE'] = df_hw["DATE"].where(df_hw["HOUR"] != 0, df_hw["DATE"] - pd.Timedelta(days=1)) # convert midnight after headway as in R code
    df_hw["HOUR"] = df_hw["HOUR"].where(df_hw["HOUR"] != 0, 24)

    return df_hw

In [12]:
# Calculate all headways for the 3 stops
carmack_2_stop = 403
carmack_3_stop = 404
transportation_hub_stop = 405

carmack_2_headways = calculate_headways(mc_busstate, carmack_2_stop)
carmack_3_headways = calculate_headways(mc_busstate, carmack_3_stop)
transport_hub_headways = calculate_headways(mc_busstate, transportation_hub_stop)

In [14]:
#transport_hub_headways.tail()
#sum(carmack_3_headways['HEADWAY'].isna()) 
# all 2 have 27 NA values for headway - I THINK this has to do with the midnight calculations - will revisit later

# NOTE: At this point is where the hours calculation can be changed - will revisit later to get a better view, for now keeping the same as it was

In [15]:

headway_df = pd.concat([carmack_2_headways, carmack_3_headways, transport_hub_headways], ignore_index=True)

# init cols - same as R
headway_df['TIME'] = "5-6a"
headway_df['TARGET'] = 3

# Set time frames and headways with ~10 minutes on either end for adjustment 
# same method from the legacy R code
time_frames = [
    (headway_df['HOUR'] <= 5),

    (headway_df['ARRIVAL'] >= 370) & (headway_df['ARRIVAL'] < 411),

    (headway_df['ARRIVAL'] >= 430) & (headway_df['ARRIVAL'] < 471),

    (headway_df["ARRIVAL"] >= 490) & (headway_df["ARRIVAL"] < 831),

    (headway_df["ARRIVAL"] >= 850) & (headway_df["ARRIVAL"] < 1190),

    (headway_df["ARRIVAL"] >= 1210) & (headway_df["ARRIVAL"] < 1310),

    (headway_df["ARRIVAL"] >= 1330) & (headway_df["ARRIVAL"] < 1431)
]

# Use same time labels as R code
time_labels = [
    "5-6a",
    "6-7a",
    "7-8a",
    "8a-2p",
    "2-8p",
    "8-10p",
    "10p-12a"
]

target_times = [ # the target headway for each timeframe
    10,  # <=5
    3,   # 6-7a
    3,   # 7-8a
    10,  # 8a-2p
    5,   # 2-8p
    10,  # 8-10p
    5    # 10p-12a
]

# Apply time buckets
headway_df['TIME'] = np.select(time_frames, time_labels, default=pd.NA)

# Apply targets
headway_df['TARGET'] = np.select(time_frames, target_times, default=pd.NA)

# filter out missing
headway_df = headway_df.loc[headway_df['TIME'].notna()].copy()

# create metric flag, converting headway / target as int
headway_df['MET'] = (headway_df['HEADWAY'] <= headway_df['TARGET']).astype(int)

headway_summary = (
    headway_df.groupby("TIME")
    .agg(
        MET=("MET", "sum"),
        COUNT=("HEADWAY", "size"),
        p50=("HEADWAY", lambda x: x.quantile(0.5)),
        p75=("HEADWAY", lambda x: x.quantile(0.75)),
        p90=("HEADWAY", lambda x: x.quantile(0.9))
    )
    .reset_index()
)

headway_summary['MET'] = headway_summary['MET'] / headway_summary['COUNT']

headway_time_order = ['5-6a', '6-7a', '7-8a', '8a-2p', '2-8p', '8-10p', '10p-12a']
headway_summary['TIME'] = pd.Categorical(headway_summary['TIME'], categories=headway_time_order, ordered=True)

headway_summary = headway_summary.sort_values('TIME').drop(columns=['COUNT'])

c:\Users\morgan.1461\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\core\computation\expressions.py:74: RuntimeWarning: invalid value encountered in less_equal
  return op(left_op, right_op)


In [16]:
headway_summary

# save headways into a csv like in R code workflow
headway_summary.to_csv(f'{dashboard_dir}/1-Headway-{month_full}-{year_full}.csv')

**FILTER FOR RIDERSHIP - capacity metric 2**

In [17]:
# this will need to be updated when stops change
mc_rider_df = mc_busstate.loc[mc_busstate['STOP_NAME'] == 'JOHN HERRICK LOOP']

# legacy R code had bus 1705 singled out - will comment out and then revisit if necessary
# mc_rider_df = mc_rider_df[~((mc_rider_df['BUS_ID'] == 1705) & ((mc_rider_df['BOARDINGS'] > 60) | (mc_rider_df['ALIGHTINGS'] > 60)))]

# load as max of boarding or alighitngs at the thub
mc_rider_df['LOAD'] = mc_rider_df[['BOARDINGS', 'ALIGHTINGS']].max(axis=1)

# assign time frames again
def assign_time_bucket(row):
    if row['HOUR'] <= 5:
        return '5-6a'
    elif row['HOUR'] == 6:
        return '6-7a'
    elif row['HOUR'] == 7:
        return '7-8a'
    elif 8 <= row['HOUR'] < 14:
        return '8a-2p'
    elif 14 <= row['HOUR'] < 20:
        return '2-8p'
    elif 20 <= row['HOUR'] < 22:
        return '8-10p'
    else:
        return '10p-12a'

mc_rider_df['TIME'] = mc_rider_df.apply(assign_time_bucket, axis=1)
mc_loads_time = mc_rider_df.groupby('TIME').size().reset_index(name='LOOPS')

# Assign categories as per R code
def assign_load_category(load):
    if load <= 30:
        return '0-30 Passengers'
    elif load <= 50:
        return '31-50 Passengers'
    elif load <= 65:
        return '51-65 Passengers'
    else:
        return '66+ Passengers'
    
mc_rider_df['SIZE'] = mc_rider_df['LOAD'].apply(assign_load_category)

# assign by hour and load cat
mc_loads_pax = (
    mc_rider_df
    .groupby(['HOUR', 'SIZE'])
    .size()
    .reset_index(name='LOOPS')
)

mc_loads_pax_size = mc_loads_pax.pivot_table(
    index='HOUR',
    columns='SIZE',
    values='LOOPS',
    fill_value=0
).reset_index()

mc_loads_pax_size['TIME'] = mc_loads_pax_size['HOUR'].apply(lambda f: assign_time_bucket({'HOUR': f}))

mc_loads_pax_size_sum = mc_loads_pax_size.groupby('TIME').sum().reset_index()

for col in ['0-30 Passengers', '31-50 Passengers', '51-65 Passengers', '66+ Passengers']:
    if col not in mc_loads_pax_size_sum.columns:
        mc_loads_pax_size_sum[col] = 0

mc_loads = mc_loads_time.merge(mc_loads_pax_size_sum, on = 'TIME', how = 'inner')

capacity_time_order = ['5-6a', '6-7a', '7-8a', '8a-2p', '2-8p', '8-10p', '10p-12a']
mc_loads['TIME'] = pd.Categorical(mc_loads['TIME'], categories=capacity_time_order, ordered=True)
mc_loads = mc_loads.drop(columns = ['HOUR'])
mc_loads = mc_loads.sort_values('TIME')



In [18]:
mc_loads.to_csv(f'{dashboard_dir}/2-Capacity-{month_full}-{year_full}.csv')

**METRIC 3 - TRAVEL TIME**

In [ ]:
# Metric 3: Travel Time (Python version of legacy R logic)

# Common time bins used in legacy ridecheck/runtime metric
ridecheck_labels = ['5:30-7a', '7-10a', '10a-4p', '4-7p', '7p-12a', '12-5a']
time_order_rt = ['5:30-7a', '7-10a', '10a-4p', '4-7p', '7p-12a', '12-5a']

def assign_ridecheck_time(partial_hour_series):
    conditions = [
        (partial_hour_series >= 5.5) & (partial_hour_series < 7),
        (partial_hour_series >= 7) & (partial_hour_series < 10),
        (partial_hour_series >= 10) & (partial_hour_series < 16),
        (partial_hour_series >= 16) & (partial_hour_series < 19),
        (partial_hour_series >= 19),
    ]
    return np.select(conditions, ridecheck_labels[:-1], default='12-5a')

# ---- Loop counts (legacy MC.loads.ridecheck) ----
mc_loads_ridecheck = mc_rider_df.copy()
mc_loads_ridecheck['HOUR_PARTIAL'] = mc_loads_ridecheck['HOUR'] + (mc_loads_ridecheck['MIN'] / 60)
mc_loads_ridecheck['TIME'] = assign_ridecheck_time(mc_loads_ridecheck['HOUR_PARTIAL'])

mc_loads_ridecheck = (
    mc_loads_ridecheck
    .groupby('TIME', as_index=False)
    .size()
    .rename(columns={'size': 'LOOPS'})
)

# Keep all categories and order
mc_loads_ridecheck['TIME'] = pd.Categorical(mc_loads_ridecheck['TIME'], categories=time_order_rt, ordered=True)
mc_loads_ridecheck = (
    mc_loads_ridecheck
    .set_index('TIME')
    .reindex(time_order_rt, fill_value=0)
    .reset_index()
)

# Runtime by leg (legacy MC.RT / MC.RT.grouped) 
mc_rt = (
    mc_busstate
    .sort_values(['BUS_ID', 'DATE', 'ARRIVAL'])
    .loc[~mc_busstate['STOP'].isin([94, 95, 404])]
    .copy()
)

# lag values within bus/day
mc_rt['PREV_DEPARTURE'] = mc_rt.groupby(['BUS_ID', 'DATE'])['DEPARTURE'].shift(1)
mc_rt['PREV_STOP_NAME'] = mc_rt.groupby(['BUS_ID', 'DATE'])['STOP_NAME'].shift(1)

mc_rt['RUN_TIME'] = mc_rt['ARRIVAL'] - mc_rt['PREV_DEPARTURE']
mc_rt['LEG'] = mc_rt['PREV_STOP_NAME'] + " - " + mc_rt['STOP_NAME']

valid_legs = ['CARMACK 2 - JOHN HERRICK LOOP', 'JOHN HERRICK LOOP - CARMACK 2']
mc_rt = mc_rt.loc[
    mc_rt['LEG'].isin(valid_legs) &
    mc_rt['RUN_TIME'].notna() &
    (mc_rt['RUN_TIME'] >= 0) &
    (mc_rt['RUN_TIME'] < 20)
].copy()

mc_rt['HOUR_PARTIAL'] = mc_rt['HOUR'] + (mc_rt['MIN'] / 60)
mc_rt['TIME'] = assign_ridecheck_time(mc_rt['HOUR_PARTIAL'])

mc_rt_grouped = (
    mc_rt
    .groupby(['TIME', 'LEG'], as_index=False)['RUN_TIME']
    .mean()
)

mc_rt_grouped['RUN_TIME'] = mc_rt_grouped['RUN_TIME'].round(1)

mc_rt_grouped = (
    mc_rt_grouped
    .pivot(index='TIME', columns='LEG', values='RUN_TIME')
    .reset_index()
)

# final metric table
metric3_travel_time = (
    mc_loads_ridecheck
    .merge(mc_rt_grouped, on='TIME', how='left')
)

metric3_travel_time['TIME'] = pd.Categorical(metric3_travel_time['TIME'], categories=time_order_rt, ordered=True)
metric3_travel_time = metric3_travel_time.sort_values('TIME')

#metric3_travel_time

In [20]:
metric3_travel_time.to_csv(f'{dashboard_dir}/3-TravelTime-{month_full}-{year_full}.csv', index=False)